In [5]:
import requests
import os
from typing import List
import pickle
import zipfile
import rasterio
from affine import Affine
import pyproj
import numpy as np
import scipy.ndimage
from rasterio.warp import reproject, Resampling
import PIL
import matplotlib.pyplot as plt
from base64 import b64encode
import tables
from pathlib import Path
try:
    from StringIO import StringIO
    py3 = False
except ImportError:
    from io import StringIO, BytesIO
    py3 = True
from ipyleaflet import Map, ImageOverlay, basemap_to_tiles, basemaps, LayerGroup, FullScreenControl, Marker, Popup, LayersControl, LegendControl
from ipywidgets import HTML

from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors

# Step 1: List of hex color values
hex_colors = ['#000000', '#006400', '#ffbb22', '#ffff4c', '#f096ff', '#fa0000', '#b4b4b4', '#f0f0f0','#0064c8', '#0096a0', '#00cf75', '#fae6a0']  # ESA world cover

# Step 2: Convert hex to RGB
rgb_colors = [mcolors.hex2color(color) for color in hex_colors]

# Step 3: Create a LinearSegmentedColormap
wc_colormap = mcolors.ListedColormap(rgb_colors, name="esa_world_cover")

## Decoding RasterSpec

In [8]:
def decode_spec(spec_arr):
    spec_bytes = spec_arr.tobytes()
    return pickle.loads(spec_bytes)

## The original data is in the WGS 84 projection, but Leaflet uses Web Mercator, so we need to reproject.

In [9]:
def reproject_to_mercator(arrs, spec):
    with rasterio.Env():
        if not isinstance(arrs, List):
            arrs = [arrs]
        rows, cols = arrs[0].shape[-2:]
        src_transform = spec.transform
        src_crs = {'init': f'EPSG:{spec.epsg}'}

        dst_crs = {'init': 'EPSG:3857'}
        dst_transform, width, height = rasterio.warp.calculate_default_transform(src_crs, dst_crs, cols, rows, *spec.bounds)
        reprojected = []
        for arr in arrs:
            if len(arr.shape) > 2:
                dst_shape = arr.shape[0], height, width
            else: 
                dst_shape = height, width
            destination = np.zeros(dst_shape)

            reproject(
                arr,
                destination,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)
            reprojected.append(destination)
    return reprojected

## convert NumPy array to an image.

In [10]:
def to_PIL(arr, minv=0, maxv=None, cm_func=None):
    if arr.shape[0] == 3: # RGB
        arr = arr.transpose((1,2,0))
        arr_mask = np.where(np.isfinite(arr[:,:,0]), 255, 0)
    else: # wc or slope
        arr_mask = np.where(np.isfinite(arr), 255, 0)
    arr_norm = (arr - minv) / (maxv - minv)
    arr_norm = np.where(np.isfinite(arr), arr_norm, 0)
    if len(arr.shape) == 2:
        arr_norm = cm_func(arr_norm)
    arr_im = PIL.Image.fromarray(np.uint8(arr_norm*255))
    mask = PIL.Image.fromarray(np.uint8(arr_mask), mode='L')
    im = PIL.Image.new('RGBA', arr_norm.shape[:2], color=None)
    im.paste(arr_im, mask=mask)
    return im

## Generate Image url

In [11]:
def get_url(img, py3=None):
    if py3:
        f = BytesIO()
    else:
        f = StringIO()
    img.save(f, 'png')
    data = b64encode(f.getvalue())
    if py3:
        data = data.decode('ascii')
    imgurl = 'data:image/png;base64,' + data
    return imgurl

In [12]:
def reproject_bounds(raster_spec, crs_to='EPSG:4326'):
    transformer = pyproj.Transformer.from_crs(f'EPSG:{raster_spec.epsg}', crs_to, always_xy=True)
    # Transform the bounds
    minx, miny = transformer.transform(raster_spec.bounds[0], raster_spec.bounds[1])
    maxx, maxy = transformer.transform(raster_spec.bounds[2], raster_spec.bounds[3])
    return ((miny,minx), (maxy, maxx))

In [13]:
def create_base_map(center = [-10, -60], zoom=2):
    m = Map(center=center, zoom=zoom, interpolation='nearest', scroll_wheel_zoom=True)
    m.layout.height = '800px'
    tile = basemap_to_tiles(basemaps.Esri.WorldImagery)
    m.add(tile)
    m.add(FullScreenControl)
    return m

In [14]:
def create_img_overlay(data, partition_number=0, year=2019):
    partition = data.root[f'{year}/{partition_number}/input'][:, [1,2,3,13]]
    s2_arr = partition[:, :3]
    wc_arr = partition[:, -1]
    slope_arr = data.root[f'{year}/{partition_number}/slope']
    spec_arr = data.root[f'{year}/{partition_number}/spec']
    loc_arr = data.root[f'{year}/{partition_number}/gedi_attrs'][:, 10:12]
    dc_arr = data.root[f'{year}/{partition_number}/defective_cover']
    dd_arr = data.root[f'{year}/{partition_number}/delta_day']
    s2_ios, wc_ios, slope_ios, loc_makers = [],[],[],[]
    for i, (s2, wc, slope) in enumerate(zip(s2_arr, wc_arr, slope_arr)):
        spec = decode_spec(spec_arr[i])
        s2, wc, slope = reproject_to_mercator([s2, wc, slope], spec)
        bounds = reproject_bounds(spec)
        img_overlays = []
        s2_img = to_PIL(s2, np.nanmin(s2), np.nanmax(s2))
        wc_img = to_PIL(wc, 0, 100, wc_colormap)
        slop_img = to_PIL(slope, 0, 90, plt.cm.gist_earth)
        for im in [s2_img, wc_img, slop_img]:
            imgurl = get_url(im, py3=py3)
            io = ImageOverlay(url=imgurl, bounds=bounds)
            img_overlays.append(io)
        s2_ios.append(img_overlays[0])
        wc_ios.append(img_overlays[1])
        slope_ios.append(img_overlays[2])
        marker = Marker(location=loc_arr[i].tolist())
        message = HTML()
        message.value = f"defective cover: {dc_arr[i]} <br>delta days: {dd_arr[i]} "
        marker.popup = message
        loc_makers.append(marker)
    return s2_ios, wc_ios, slope_ios, loc_makers

In [15]:
data_dir = 'data/GEDI/vis'
zone = '01G'
h5_file = Path.home() / data_dir/ f'{zone}.h5'
data = tables.open_file(h5_file)

In [ ]:
layers_list = create_img_overlay(data, 7)

In [17]:
m = create_base_map()
for i, layers in enumerate(layers_list):
    group = LayerGroup(name=f'group{i}',layers=layers)
    m.add(group)
control = LayersControl(position='topright')
m.add(control)
# overlay_group.interact(opacity=(0.0,1.0,0.01))

Map(center=[-10, -60], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

In [18]:
m.layout.height = '800px'

In [29]:
import os
import geopandas as gpd
mgrs_df = gpd.read_parquet('~/GEDI/mgrs_with_count_orbits_and_sampled.parquet')


In [30]:
import os 
ns = len(small_zones)
for year in range(2019, 2023):
    wanted_zones = mgrs_df[mgrs_df[f'sampled_{year}']>0].MGRS_UTM
    processed_zones = os.listdir(f'/users/zhanghui/scratch/S2_geoparquet_items/{year}')
    processed_zones = [z[:3] for z in processed_zones]
    print(len(wanted_zones), len(processed_zones), ns)
    if len(wanted_zones) - len(processed_zones) > ns:
        unprecessed = set(wanted_zones) - set(processed_zones) - set(small_zones)
        print(unprecessed)


444 320 138
446 319 138
456 320 138
453 321 138


In [ ]:
'S2A_MSIL2A_20210930T071211_N0300_R020_T38KLV_20211001T181115'

In [32]:
import planetary_computer
import pystac_client
import pystac
import pandas as pd
import adlfs

stac_endpoint = 'https://planetarycomputer.microsoft.com/api/stac/v1'
api = pystac_client.Client.open(stac_endpoint, modifier=planetary_computer.sign_inplace)
s2asset = api.get_collection("sentinel-2-l2a").assets["geoparquet-items"]
fs = adlfs.AzureBlobFileSystem(
    **s2asset.extra_fields["table:storage_options"]).ls("items/sentinel-2-l2a.parquet")
fs_df = pd.DataFrame(fs, columns=['fname'])
date_range = fs_df.fname.str.findall(r'\d{4}-\d{2}-\d{2}')
fs_df['start'] = pd.to_datetime(date_range.str[0])
fs_df['end'] = pd.to_datetime(date_range.str[1])
fs_df['fname'] = 'abfs://' + fs_df['fname']

start = pd.Timestamp('2022-01-17')
end = pd.Timestamp('2022-01-19')
filtered = fs_df[(fs_df['start'] < end) & (fs_df['end'] > start)]
files = filtered['fname'].to_list()

In [35]:
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
from azure.identity import DefaultAzureCredential
connection_string = 'YweGsNV6EZqQeSB65D8ro40QEMGnyqx3sGDiFKtHDL5pOpLGWEeCbymzyJ2cxa5Zz83sBY6535c1+ASt7FozHw=='
# service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = 'sentinel2-l2'
account_url = "https://sentinel2l2a01.blob.core.windows.net"
default_credential = DefaultAzureCredential()

# Create the BlobServiceClient object
blob_service_client = BlobServiceClient(account_url, credential=default_credential)
container_client = blob_service_client.get_container_client(container_name)
blobs_list = container_client.list_blobs()
for blob in blobs_list:
    print(blob.name)
    break

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: PowerShell is not installed
	AzureDeveloperCliCredential: Azure Developer CLI could not be found. Please visit https://aka.ms/azure-dev for installation instructions and then,once installed, authenticate to your Azure account using 'azd auth login'.
To mitigate this issue, please refer to the troubleshooting guidelines here at https://aka.ms/azsdk/py

KeyboardInterrupt: 

<iterator object azure.core.paging.ItemPaged at 0x14c961988e20>

In [51]:
import planetary_computer
import pystac_client
import pystac
import pandas as pd
import adlfs

stac_endpoint = 'https://planetarycomputer.microsoft.com/api/stac/v1'
api = pystac_client.Client.open(stac_endpoint, modifier=planetary_computer.sign_inplace)
s2asset = api.get_collection("sentinel-2-l2a").assets["geoparquet-items"]
fs = adlfs.AzureBlobFileSystem(
    **s2asset.extra_fields["table:storage_options"]).ls("items/sentinel-2-l2a.parquet")
fs_df = pd.DataFrame(fs, columns=['fname'])
date_range = fs_df.fname.str.findall(r'\d{4}-\d{2}-\d{2}')
fs_df['start'] = pd.to_datetime(date_range.str[0])
fs_df['end'] = pd.to_datetime(date_range.str[1])
fs_df['fname'] = 'abfs://' + fs_df['fname']
date = '2022-01-18'
start = pd.Timestamp('2022-01-18')
end = pd.Timestamp('2022-01-18')
filtered = fs_df[(fs_df['start'] < end) & (fs_df['end'] > start)]
files = filtered['fname'].to_list()

In [52]:
import dask_geopandas as dgp
mgrs_tiles = ['35QRE']
s2_df = dgp.read_parquet(
    files,
    storage_options=s2asset.extra_fields["table:storage_options"],
    gather_spatial_partitions=False,
    columns=['id', 'geometry', 'eo:cloud_cover'],
    filters=[('s2:mgrs_tile', 'in', mgrs_tiles),
            ("eo:cloud_cover", "<", 50),
            ('s2:water_percentage', '<', 99)]
)
s2_df = s2_df.map_partitions(lambda x: x, meta=s2_df).compute()

In [53]:
s2_df.id.to_list()

['S2A_MSIL2A_20220118T083301_R021_T35QRE_20220119T070207',
 'S2B_MSIL2A_20220123T083139_R021_T35QRE_20220123T233201']

In [1]:
import geopandas as gpd
gdf = gpd.read_parquet('~/scratch/GEDI_with_s2_candidates_and_best/2021/19H/partition_158.parquet')
ids = gdf['s2_candidates'].explode().drop_duplicates()
ids[ids=='S2B_MSIL2A_20220128T143719_R096_T19HDB_20220213T084044']

32    S2B_MSIL2A_20220128T143719_R096_T19HDB_2022021...
Name: s2_candidates, dtype: object

In [1]:
import planetary_computer
import pystac_client
import pystac
import pandas as pd
import adlfs

stac_endpoint = 'https://planetarycomputer.microsoft.com/api/stac/v1'
tile_id = 'S2B_MSIL2A_20220123T083139_R021_T35QRE_20220123T233201'
api = pystac_client.Client.open(stac_endpoint, modifier=planetary_computer.sign_inplace)

search = api.search(collections=['sentinel-2-l2a'],
                    query={
                        "s2:mgrs_tile": {
                            "eq": '35QRE'
                        }
                    },
                    datetime=f'2022-01-03/2022-02-27'
                    )
items = search.item_collection()
items
# url = f'{stac_endpoint}/collections/sentinel-2-l2a/items/{tile_id}'
# item = pystac.Item.from_file(url)
# item = planetary_computer.sign(item)
# item

In [3]:
[(i.id, i.properties['eo:cloud_cover']) for i in items.items]

[('S2A_MSIL2A_20220227T082911_R021_T35QRE_20220302T203614', 0.001347),
 ('S2B_MSIL2A_20220222T082839_R021_T35QRE_20220228T123831', 0.612158),
 ('S2A_MSIL2A_20220217T083021_R021_T35QRE_20220224T115212', 1.513144),
 ('S2B_MSIL2A_20220212T082939_R021_T35QRE_20220222T061721', 6.516289),
 ('S2A_MSIL2A_20220207T083121_R021_T35QRE_20220220T002110', 0.105159),
 ('S2B_MSIL2A_20220202T083049_R021_T35QRE_20220217T134803', 0.002193),
 ('S2A_MSIL2A_20220128T083221_R021_T35QRE_20220213T052200', 2.801517),
 ('S2B_MSIL2A_20220123T083139_R021_T35QRE_20220123T233201', 1.919589),
 ('S2A_MSIL2A_20220118T083301_R021_T35QRE_20220119T070207', 0.058507),
 ('S2B_MSIL2A_20220113T083219_R021_T35QRE_20220113T193301', 22.553081),
 ('S2A_MSIL2A_20220108T083331_R021_T35QRE_20220110T145809', 96.207653),
 ('S2B_MSIL2A_20220103T083239_R021_T35QRE_20220103T234825', 0.092034)]

In [24]:
from utils._stackstac import stack
image = stack(item, assets=['SCL'], resolution=20, epsg=32722, dtype='uint8', fill_value=0)
image

<xarray.DataArray 'stackstac-f3b4d34e45e00c88c683219d9a8c8921' (time: 1,
                                                                band: 1,
                                                                y: 5490, x: 5490)>
dask.array<fetch_raster_window, shape=(1, 1, 5490, 5490), dtype=uint8, chunksize=(1, 1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates: (12/43)
  * time                                     (time) datetime64[ns] 2022-11-11...
    id                                       (time) <U54 'S2A_MSIL2A_20221111...
  * band                                     (band) <U3 'SCL'
  * x                                        (x) float64 8e+05 ... 9.098e+05
  * y                                        (y) float64 9.8e+06 ... 9.69e+06
    s2:reflectance_conversion_factor         float64 1.019
    ...                                       ...
    proj:bbox                                object {799980.0, 909780.0, 9690...
    proj:transform                           object {0.0, 799980.0, -20.0, 20...
    proj:shape                               object {5490}
    title                                    <U29 'Scene classfication map (S...
    gsd                                      float64 20.0
    epsg                                     int64 32722
Attributes:
    spec:        RasterSpec(epsg=32722, bounds=(799980, 9690220, 909780, 9800...
    crs:         epsg:32722
    transform:   | 20.00, 0.00, 799980.00|\n| 0.00,-20.00, 9800020.00|\n| 0.0...
    resolution:  20

In [22]:
try:
    image = image.load()
except Exception as e:
    print(e)
    print(e.error_message)

Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/22/M/HC/2022/11/11/S2A_MSIL2A_20221111T133841_N0400_R124_T22MHC_20221112T031803.SAFE/GRANULE/L2A_T22MHC_A038588_20221111T133838/IMG_DATA/R20m/T22MHC_20221111T133841_SCL_20m.tif?st=2024-04-10T20%3A18%3A28Z&se=2024-04-11T21%3A03%3A28Z&sp=rl&sv=2021-06-08&sr=c&skoid=c85c15d6-d1ae-42d4-af60-e2ca0f81359b&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2024-04-11T08%3A28%3A39Z&ske=2024-04-18T08%3A28%3A39Z&sks=b&skv=2021-06-08&sig=HkDmtSvc5kndp6r2rRbrva6MtF5S1R09UG%2BYd9LsMDk%3D': RasterioIOError("'/vsicurl/https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/22/M/HC/2022/11/11/S2A_MSIL2A_20221111T133841_N0400_R124_T22MHC_20221112T031803.SAFE/GRANULE/L2A_T22MHC_A038588_20221111T133838/IMG_DATA/R20m/T22MHC_20221111T133841_SCL_20m.tif?st=2024-04-10T20%3A18%3A28Z&se=2024-04-11T21%3A03%3A28Z&sp=rl&sv=2021-06-08&sr=c&skoid=c85c15d6-d1ae-42d4-af60-e2ca0f81359b&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2024-04-11T08%3A

AttributeError: 'RuntimeError' object has no attribute 'error_message'

In [25]:
image = image.load()

RuntimeError: Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/22/M/HC/2022/11/11/S2A_MSIL2A_20221111T133841_N0400_R124_T22MHC_20221112T031803.SAFE/GRANULE/L2A_T22MHC_A038588_20221111T133838/IMG_DATA/R20m/T22MHC_20221111T133841_SCL_20m.tif?st=2024-04-10T20%3A18%3A28Z&se=2024-04-11T21%3A03%3A28Z&sp=rl&sv=2021-06-08&sr=c&skoid=c85c15d6-d1ae-42d4-af60-e2ca0f81359b&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2024-04-11T08%3A28%3A39Z&ske=2024-04-18T08%3A28%3A39Z&sks=b&skv=2021-06-08&sig=HkDmtSvc5kndp6r2rRbrva6MtF5S1R09UG%2BYd9LsMDk%3D': RasterioIOError("'/vsicurl/https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/22/M/HC/2022/11/11/S2A_MSIL2A_20221111T133841_N0400_R124_T22MHC_20221112T031803.SAFE/GRANULE/L2A_T22MHC_A038588_20221111T133838/IMG_DATA/R20m/T22MHC_20221111T133841_SCL_20m.tif?st=2024-04-10T20%3A18%3A28Z&se=2024-04-11T21%3A03%3A28Z&sp=rl&sv=2021-06-08&sr=c&skoid=c85c15d6-d1ae-42d4-af60-e2ca0f81359b&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=20

In [5]:
import geopandas as gpd
df = gpd.read_parquet('/users/zhanghui/scratch/GEDI_with_s2_candidates_and_best/2019/40M/partition_0.parquet')
df

,geometry,beam,delta_time,digital_elevation_model,digital_elevation_model_srtm,elev_highestreturn,elev_lowestmode,elevation_bias_flag,energy_total,landsat_treecover,...,rh96,rh97,rh98,rh99,rh100,date,s2_candidates,best_s2,delta_day,defective_cover
0,POINT (55.48478 -4.75860),3,4.257398e+07,-35.101883,-37.110424,-30.533535,-37.794250,0,7057.118164,0.0,...,5.16,5.42,5.72,6.21,7.26,2019-05-08,[S2A_MSIL2A_20190316T064031_R034_T40MCV_202010...,S2B_MSIL2A_20190510T064049_R034_T40MCV_2020110...,2.0,0.0
1,POINT (55.46293 -4.77057),1,4.257398e+07,-999999.000000,-999999.000000,-36.937428,-39.932236,1,2369.300293,0.0,...,2.24,2.35,2.54,2.73,2.99,2019-05-08,[S2A_MSIL2A_20190316T064031_R034_T40MCV_202010...,S2B_MSIL2A_20190510T064049_R034_T40MCV_2020110...,2.0,0.0
2,POINT (55.45603 -4.78024),1,4.257398e+07,-999999.000000,-999999.000000,-36.941036,-40.123020,1,1676.568115,0.0,...,2.24,2.39,2.58,2.84,3.18,2019-05-08,[S2A_MSIL2A_20190316T064031_R034_T40MCV_202010...,S2B_MSIL2A_20190510T064049_R034_T40MCV_2020110...,2.0,0.0
3,POINT (55.45714 -4.78774),2,4.257398e+07,-999999.000000,-999999.000000,-37.440838,-40.060753,0,1028.370117,0.0,...,2.13,2.20,2.32,2.47,2.61,2019-05-08,[S2A_MSIL2A_20190316T064031_R034_T40MCV_202010...,S2B_MSIL2A_20190510T064049_R034_T40MCV_2020110...,2.0,0.0
4,POINT (55.45094 -4.78740),1,4.257398e+07,-999999.000000,-999999.000000,-38.463253,-40.634491,1,530.655579,0.0,...,1.87,1.90,1.98,2.05,2.17,2019-05-08,[S2A_MSIL2A_20190316T064031_R034_T40MCV_202010...,S2B_MSIL2A_20190510T064049_R034_T40MCV_2020110...,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,POINT (55.25726 -4.51675),11,5.876744e+07,-999999.000000,-999999.000000,-35.529530,-39.329216,0,2707.622803,0.0,...,2.64,2.83,3.05,3.35,3.79,2019-11-12,[S2B_MSIL2A_20191229T065029_R077_T40MCA_202010...,S2A_MSIL2A_20191214T065031_R077_T40MCA_2020100...,32.0,0.0
207,POINT (55.47979 -4.76887),1,5.876745e+07,-999999.000000,-999999.000000,-36.727631,-39.486519,1,1188.545166,0.0,...,2.27,2.34,2.46,2.60,2.75,2019-11-12,[S2A_MSIL2A_20190823T064041_R034_T40MCV_202010...,S2A_MSIL2A_20191111T064041_R034_T40MCV_2020100...,1.0,0.0
208,POINT (55.51576 -4.81016),0,5.876745e+07,-999999.000000,-999999.000000,-35.913765,-38.747726,1,1427.394775,0.0,...,2.23,2.34,2.49,2.64,2.83,2019-11-12,[S2A_MSIL2A_20190823T064041_R034_T40MCV_202010...,S2A_MSIL2A_20191111T064041_R034_T40MCV_2020100...,1.0,0.0
209,POINT (55.51175 -4.81388),1,5.876745e+07,-999999.000000,-999999.000000,-36.703400,-39.685982,1,1731.407227,0.0,...,2.05,2.19,2.34,2.57,2.98,2019-11-12,[S2A_MSIL2A_20190823T064041_R034_T40MCV_202010...,S2B_MSIL2A_20191106T064039_R034_T40MCV_2020100...,6.0,0.0


## find zone not complete with finding best s2

In [6]:
import os
from pathlib import Path
small_zones=['23J','25L','25S','26K','26P','26Q','26S','27P','27Q','27R','28H','28M','30K','30L','31M','32K','32L','33H','38M','60U','40K','40M','40P','41K','42M','43M','43N','46N','46P','48L','49J','49K','49L','50P','52P','52R','53N','54N','54P','54R','55N','55P','55Q','55T','55U','56K','56L','56N','56P','56R','56T','56U','57J','57K','57L','57M','57N','57U','58J','58K','58L','58M','58P','58Q','59H','59K','59L','59N','60K','60M','01K','01L','01R','02K','02L','03K','03L','04K','04L','04N','05K','05L','05M','06J','06K','06L','07K','07L','07M','08K','40L','58N','59P','60L','01J','01N','02M','04M','07J','29G','58G','60G','01G','02Q','02R','28N','22T','22U','28S','29U','01U','02U','04Q','05Q','52H','20S','54G','37G','58F','60F','09J','10J','11Q','12J','13J','15M','15N','16M','17L','18K','21F','20F','21P','14P','12Q','18R','20Q','21T']
data_dir = Path('/users/zhanghui/scratch/GEDI_with_s2_candidates_and_best')
unfinished = []
for year in range(2019, 2023):
    flags = list(data_dir.glob(f'{year}/*_done'))
    finished_zones = [f.stem.split('_')[0] for f in flags]
    zones = [f.stem for f in data_dir.glob(f'{year}/*') if not f.stem.endswith('done')]
    unfinished_ = [z for z in zones if (z not in finished_zones) and (z not in small_zones)]
    unfinished.append(unfinished_)
print(unfinished)


[[], [], [], []]


In [4]:
import os
unfinished = [['36Q', '24K', '41T', '46U', '37U', '13T', '48R', '46R', '33N'], ['36Q', '35H', '31U', '24L', '22M', '15R', '43T', '24K', '29N', '41T', '46U', '37U', '19L', '45R', '33L', '44S', '46R', '51J', '23M', '15S', '20H', '20K', '21H']]
with_best_dir = '/users/zhanghui/scratch/GEDI_with_s2_candidates_and_best/'
candidates_dir = '/users/zhanghui/flash/GEDI_with_s2_candidates/'
for i, year in enumerate(range(2021, 2023)):
    for zone in unfinished[i]:
        finished_partitions = len(list(os.listdir(f'{with_best_dir}/{year}/{zone}')))
        total_partitions = len(list(os.listdir(f'{candidates_dir}/{year}/{zone}')))
        print(f'unfinished number of partitions: {total_partitions - finished_partitions}, {year} {zone}')

unfinished number of partitions: 16, 2021 36Q
unfinished number of partitions: 2, 2021 24K
unfinished number of partitions: 6, 2021 41T
unfinished number of partitions: 8, 2021 46U
unfinished number of partitions: 1, 2021 37U
unfinished number of partitions: 4, 2021 13T
unfinished number of partitions: 522, 2021 48R
unfinished number of partitions: 14, 2021 46R
unfinished number of partitions: 352, 2021 33N
unfinished number of partitions: 12, 2022 36Q
unfinished number of partitions: 3, 2022 35H
unfinished number of partitions: 43, 2022 31U
unfinished number of partitions: 5, 2022 24L
unfinished number of partitions: 18, 2022 22M
unfinished number of partitions: 9, 2022 15R
unfinished number of partitions: 11, 2022 43T
unfinished number of partitions: 8, 2022 24K
unfinished number of partitions: 6, 2022 29N
unfinished number of partitions: 25, 2022 41T
unfinished number of partitions: 21, 2022 46U
unfinished number of partitions: 14, 2022 37U
unfinished number of partitions: 25, 2022 

In [16]:
import geopandas as gpd
import dask_geopandas
df_old = gpd.read_parquet('/users/zhanghui/GEDI/mgrs_with_nbest.parquet')
df_new = gpd.read_parquet('/users/zhanghui/GEDI/mgrs_with_nbest_v1.parquet')

for year in range(2019, 2023):
    df_old[f'nbest_{year}'] = df_old[f'nbest_{year}'].combine(df_new[f'nbest_{year}'], max)
    df_old[f'enough_{year}'] = df_old[f'nbest_{year}'] >= df_old['nwant']

In [18]:
for year in range(2019, 2023):
    tot = df_old[f'nbest_{year}'].sum()
    tot_locs = df_old[f'sampled_{year}'].sum()
    print(f'sampled {tot_locs}, {tot} found with s2')

sampled 76529150, 75807502 found with s2
sampled 77309620, 76045982 found with s2
sampled 76843486, 75605334 found with s2
sampled 77145760, 75851954 found with s2


In [24]:
df = gpd.read_parquet('/users/zhanghui/GEDI/mgrs_with_nbest_v2.parquet')
df = df[['MGRS_UTM','sampled_2019', 'sampled_2020', 'sampled_2021', 'sampled_2022', 'nwant',
       'nbest_2019', 'enough_2019', 'nbest_2020', 'enough_2020', 'nbest_2021',
       'enough_2021', 'nbest_2022', 'enough_2022']]

In [32]:
df['nwant'] = df['nwant'].round()
df.to_csv('mgrs_with_nbest_v2.csv')

In [33]:
df

,MGRS_UTM,sampled_2019,sampled_2020,sampled_2021,sampled_2022,nwant,nbest_2019,enough_2019,nbest_2020,enough_2020,nbest_2021,enough_2021,nbest_2022,enough_2022
0,23J,6542,6613,6552,6668,6533.0,6542,True,6613,True,6550,True,6668,True
1,23K,372170,374088,371350,373437,365971.0,372169,True,374087,True,371327,True,373435,True
2,23L,414017,415222,414690,415017,408001.0,413949,True,415176,True,414631,True,414977,True
3,23M,332789,334783,331811,332794,327255.0,332394,True,334674,True,331394,True,332130,True
4,24K,108377,110362,109237,109322,107716.0,108364,True,110346,True,109208,True,109286,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,19Q,43352,43775,43541,43757,42483.0,43274,True,43718,True,43483,True,43742,True
454,19T,156726,157011,156799,156693,154715.0,156635,True,152898,False,154867,True,155010,True
455,20Q,3300,3453,3389,3349,3237.0,3279,True,3441,True,3379,True,3347,True
456,20T,62246,61694,61930,62309,60856.0,62235,True,61258,True,61908,True,62272,True


In [27]:
test = df[~df['enough_2019']]

In [30]:
test[test['MGRS_UTM']=='32M']

,MGRS_UTM,sampled_2019,sampled_2020,sampled_2021,sampled_2022,nwant,nbest_2019,enough_2019,nbest_2020,enough_2020,nbest_2021,enough_2021,nbest_2022,enough_2022
24,32M,83910,83360,83364,83514,82971.491335,81564,False,78588,False,73952,False,75518,False


In [28]:
test['MGRS_UTM'].to_list()

['26K',
 '26Q',
 '28H',
 '28M',
 '30K',
 '30L',
 '31M',
 '32K',
 '32L',
 '32M',
 '32N',
 '33M',
 '60U',
 '40K',
 '41K',
 '43N',
 '44R',
 '44S',
 '45R',
 '46R',
 '46S',
 '47R',
 '48R',
 '52R',
 '54M',
 '54N',
 '54P',
 '54R',
 '55N',
 '55P',
 '55Q',
 '56N',
 '56P',
 '56R',
 '56T',
 '56U',
 '57J',
 '57K',
 '57M',
 '57N',
 '57U',
 '58J',
 '58P',
 '58Q',
 '60K',
 '01K',
 '01R',
 '03L',
 '04L',
 '04N',
 '05K',
 '05M',
 '06J',
 '07K',
 '08K',
 '40L',
 '58N',
 '59P',
 '60L',
 '01J',
 '01N',
 '02M',
 '04M',
 '07J',
 '29G',
 '02Q',
 '02R',
 '30N',
 '31N',
 '28S',
 '09U',
 '43S',
 '02U',
 '54G',
 '58F',
 '60F',
 '10J',
 '11Q',
 '12J',
 '13J',
 '16M',
 '17M',
 '17P',
 '18G',
 '19H',
 '19J',
 '19K',
 '18F',
 '17N',
 '18N']

In [1]:
import geopandas as gpd
import dask_geopandas as dgp
mgrs_df = gpd.read_parquet('~/GEDI/mgrs_with_nbest.parquet')
mgrs_df


,geometry,MGRS_UTM,landmass,tracks,count_2019,count_2020,count_2021,count_2022,count_2023,tracks_2019,...,sampled_2022,nwant,nbest_2019,enough_2019,nbest_2020,enough_2020,nbest_2021,enough_2021,nbest_2022,enough_2022
0,"POLYGON ((-48.00000 -32.00000, -47.95312 -32.0...",23J,9288.419035,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911006241...,58655,66611,113603,115791,8238,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911604313...,...,6668,6533.411723,-1,False,-1,False,-1,False,-1,False
1,"POLYGON ((-48.00000 -24.00000, -47.95312 -24.0...",23K,520293.724930,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910808033...,5437176,8178846,7115938,7711686,871554,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910808033...,...,373437,365971.120494,372169,True,374087,True,371327,True,373435,True
2,"POLYGON ((-48.00000 -16.00000, -47.95312 -16.0...",23L,580047.145974,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910808033...,5533421,8389945,6259297,7102555,1433353,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910808033...,...,415017,408001.276548,413949,True,415176,True,414631,True,414977,True
3,"POLYGON ((-48.00000 -8.00000, -47.90625 -8.000...",23M,465251.706893,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910919352...,5724767,6795706,5370067,5560024,790155,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910919352...,...,332794,327254.933751,332394,True,334674,True,331394,True,-1,False
4,"POLYGON ((-42.00000 -24.00000, -41.95313 -24.0...",24K,153137.823810,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911006241...,1201636,2457871,1747315,1568132,325987,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911119284...,...,109322,107716.119346,108364,True,110346,True,-1,False,-1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
453,"POLYGON ((-72.00000 16.00000, -71.95312 16.000...",19Q,60397.903829,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910811090...,795081,1219458,1070236,1158902,244253,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911110124...,...,43757,42483.480927,43274,True,43718,True,43483,True,43742,True
454,"POLYGON ((-72.00000 40.00000, -71.95312 40.000...",19T,219954.540612,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910918024...,2120029,2824172,2031112,2204945,226584,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911813405...,...,156693,154714.550317,156635,True,152898,False,154867,True,155010,True
455,"POLYGON ((-66.00000 16.00000, -65.95313 16.000...",20Q,4601.661669,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910811090...,64426,138784,82047,70191,12438,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201911209225...,...,3349,3236.777990,-1,False,-1,False,-1,False,-1,False
456,"POLYGON ((-66.00000 48.00000, -66.00000 40.000...",20T,86517.056681,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201910817194...,1081765,1114373,1154293,1168723,194372,[LARSE/GEDI/GEDI02_A_002/GEDI02_A_201912112442...,...,62309,60855.518063,62235,True,61258,True,61908,True,62272,True
